# Introduction to NumPy

**NumPy** (Numerical Python) is the foundational library for numerical computing in Python. It provides:

- A powerful **N-dimensional array** object (`ndarray`)
- **Vectorized operations** — fast element-wise math without Python loops
- Built-in mathematical functions
- Tools for **linear algebra**, **Fourier transforms**, and **random number generation**
- Seamless integration with **Pandas**, **Matplotlib**, **SciPy**, and most of the data science ecosystem
- **C under the hood** — core operations are implemented in compiled C code, so they run much faster than equivalent pure Python loops

## Why NumPy?

| Feature | Python List | NumPy Array |
|---|---|---|
| Speed | Slow (interpreted loops) | Fast (C-backed, vectorized) |
| Memory | High overhead | Compact, typed |
| Math ops | Manual loops | Built-in, concise |
| Multi-dimensional | Nested lists | Native N-D support |

## Key Concepts

- **`ndarray`** — the core data structure; a grid of values all of the same type
- **dtype** — the data type of array elements (e.g. `int32`, `float64`)
- **shape** — a tuple describing the dimensions (e.g. `(3, 4)` for a 3×4 matrix)
- **Broadcasting** — NumPy's rules for doing arithmetic on arrays of different shapes
- **Vectorization** — replacing explicit loops with array expressions for speed and clarity

## Installation

```bash
pip install numpy
```

## Import Convention

```python
import numpy as np
```

## Why is NumPy So Fast? — Contiguous Memory Layout

The performance gap you just saw comes down to one fundamental difference in how Python lists and NumPy arrays are stored in memory.

---

### How a Python List Stores Data

A Python list does **not** store the actual values side by side. Instead, it stores a sequence of **pointers** — each pointer points to a separate Python object somewhere in memory.

```
Python list: [1, 2, 3, 4, 5]

Memory (scattered):
  list object → [ ptr0 | ptr1 | ptr2 | ptr3 | ptr4 ]
                    |       |       |       |       |
                  obj:1   obj:2   obj:3   obj:4   obj:5
                (anywhere in heap, not next to each other)
```

Each `obj` is a full Python object with its own type info, reference count, and value. To loop over the list, Python must:
1. Follow each pointer to a different memory location
2. Unbox the integer value from the Python object
3. Perform the operation
4. Re-box the result into a new Python object

This is called **pointer chasing** — jumping around in memory — which is slow because modern CPUs are optimized for reading memory in a straight line (cache lines).

---

### How a NumPy Array Stores Data

A NumPy array stores all its values **back-to-back in a single contiguous block of memory**, with no pointers and no Python object wrappers.

```
NumPy array: np.array([1, 2, 3, 4, 5], dtype=int64)

Memory (contiguous block):
  [ 1 | 2 | 3 | 4 | 5 ]
    ↑
  one flat C array, all int64 values packed together
```

Because the values are packed tightly:
- The CPU can **prefetch** the next values before they're needed (hardware prefetcher)
- Data loads into the **CPU cache** in one or two cache lines instead of many separate fetches
- NumPy hands the entire block to a **C loop** that runs without any Python overhead per element

---

### Memory Layout: Row-Major (C order)

For multi-dimensional arrays, NumPy defaults to **row-major (C) order** — rows are stored one after another in memory.

```
2D array shape (3, 4):

Logical view:         Memory (flat, row-major):
┌────────────────┐    [ a00 a01 a02 a03 | a10 a11 a12 a13 | a20 a21 a22 a23 ]
│ a00 a01 a02 a03│      ←── row 0 ───→   ←── row 1 ───→   ←── row 2 ───→
│ a10 a11 a12 a13│
│ a20 a21 a22 a23│
└────────────────┘
```

Iterating **row by row** is therefore cache-friendly. Iterating **column by column** jumps around and is slower.

---

### The strides System

NumPy tracks how to navigate the contiguous block using **strides** — the number of bytes to step in each dimension.

```python
a = np.array([[1, 2, 3],
              [4, 5, 6]], dtype=np.int64)

print(a.strides)  # (24, 8)
#  └─ 24 bytes to move to the next row (3 elements × 8 bytes)
#        └─ 8 bytes to move to the next column (1 element × 8 bytes)
```

Strides are what make **slicing free** — a slice doesn't copy data, it just returns a new view with adjusted strides and a different starting pointer into the same memory block.

---

### Summary

| | Python List | NumPy Array |
|---|---|---|
| Storage | Scattered pointers to objects | Single contiguous memory block |
| Element type | Any Python object (boxed) | Fixed C type (unboxed) |
| CPU cache use | Poor (pointer chasing) | Excellent (sequential access) |
| Loop cost | Python interpreter per element | One C loop for entire array |
| Slice | Creates new list | Zero-copy view (just new strides) |

> **The single contiguous block is the reason NumPy can hand work directly to C and BLAS routines — and why it beats Python loops by 10–100× on numerical tasks.**

## 1. Importing NumPy

By convention NumPy is always imported as `np`. This alias is used universally across the ecosystem, so any code you find online will use it too.

In [1]:
import numpy as np
import time

## 2. Creating Your First Array

`np.array()` converts a Python list into an `ndarray`. Notice the type changes from `list` to `numpy.ndarray` — this is the core NumPy data structure that all operations are built on.

In [2]:
array = np.array([1,2,3,4,5])
print(array)
print(type(array))

[1 2 3 4 5]
<class 'numpy.ndarray'>


## 3. Performance Comparison — List vs NumPy

Here we double every element in a 1-million-item collection using both a Python list comprehension and a NumPy vectorized operation.

- The **list comprehension** loops through each element in Python one by one
- The **NumPy operation** (`np_array * 2`) sends the entire array to C, which processes it in one shot

The timing output shows NumPy is roughly **10–20× faster** — and the gap widens as data size grows.

In [3]:
#execution performance
size = 10_000_000

#python list
py_list = list(range(size))
start = time.time()
sq_list = [x*2 for x in py_list]
end = time.time()
print("Python list time:", end - start)

#numpy array
np_array = np.array(py_list)
start = time.time()
#vectorized operation
sq_array = np_array * 2
end = time.time()
print("Numpy array time:", end - start)

Python list time: 0.40465593338012695
Numpy array time: 0.04459547996520996


In [4]:
#memory
import sys
print(f"Python list memory: {sys.getsizeof(sq_list)} bytes")
print(f"Python array memory: {sys.getsizeof(sq_array)} bytes")
print(f"Numpy array memory: {sq_array.nbytes} bytes")

Python list memory: 89095160 bytes
Python array memory: 80000112 bytes
Numpy array memory: 80000000 bytes


## 4.Different ways of creating Array

In [5]:
# Create from lists
arr = np.array([1, 2, 3, 4, 5])
print(arr, type(arr),arr.shape,arr.dtype)

arr2 = np.array([1,2,3,4,5,"sai"])
print(arr2, type(arr2),arr2.shape,arr2.dtype)

arr2D = np.array([[1,2,3],[4,5,6],[7,8,9]])
print(arr2D, type(arr2D),arr2D.shape,arr2D.dtype)

[1 2 3 4 5] <class 'numpy.ndarray'> (5,) int64
['1' '2' '3' '4' '5' 'sai'] <class 'numpy.ndarray'> (6,) <U21
[[1 2 3]
 [4 5 6]
 [7 8 9]] <class 'numpy.ndarray'> (3, 3) int64


In [6]:
# Create from scratch
arr_zeros = np.zeros((2, 3), dtype="int64") #prefill zeros
print(arr_zeros, type(arr_zeros),arr_zeros.shape,arr_zeros.dtype)

arr_ones = np.ones((5,), dtype="float32") #prefill ones
print(arr_ones, type(arr_ones),arr_ones.shape,arr_ones.dtype)

arr_full = np.full((3,4), 7) #prefill with a specific value
print(arr_full, type(arr_full),arr_full.shape,arr_full.dtype)

arr_eye = np.eye(3, dtype="int64") #identity matrix
print(arr_eye, type(arr_eye),arr_eye.shape,arr_eye.dtype)   

arr_range = np.arange(0, 10, 2, dtype="int64") #create an array with a range of values
print(arr_range, type(arr_range),arr_range.shape,arr_range.dtype)

arr_linspace = np.linspace(1, 100, 10, dtype="float32") #create an array with evenly spaced values
print(arr_linspace, type(arr_linspace),arr_linspace.shape,arr_linspace.dtype)

[[0 0 0]
 [0 0 0]] <class 'numpy.ndarray'> (2, 3) int64
[1. 1. 1. 1. 1.] <class 'numpy.ndarray'> (5,) float32
[[7 7 7 7]
 [7 7 7 7]
 [7 7 7 7]] <class 'numpy.ndarray'> (3, 4) int64
[[1 0 0]
 [0 1 0]
 [0 0 1]] <class 'numpy.ndarray'> (3, 3) int64
[0 2 4 6 8] <class 'numpy.ndarray'> (5,) int64
[  1.  12.  23.  34.  45.  56.  67.  78.  89. 100.] <class 'numpy.ndarray'> (10,) float32


## 5.Properties of numpy arrays

In [7]:
# Properties of numpy arrays
arr = np.array([[1, 2, 3], [4, 5, 6],[7, 8, 9]])
print(arr)
print("Shape:", arr.shape)
print("Size:", arr.size)
print("Number of dimensions:", arr.ndim)
print("Data type:", arr.dtype)

float_arr = arr.astype(np.float64) #change data type
print(float_arr, type(float_arr),float_arr.shape,float_arr.dtype)
print("Data type after astype:", float_arr.dtype)

[[1 2 3]
 [4 5 6]
 [7 8 9]]
Shape: (3, 3)
Size: 9
Number of dimensions: 2
Data type: int64
[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]] <class 'numpy.ndarray'> (3, 3) float64
Data type after astype: float64


## 6.Changing the Data Type of an Array

In [8]:
# Specify dtype at creation
str_arr = np.array([1, 2, 3], dtype="U")
print(str_arr, str_arr.dtype)

float_arr = np.array([1, 2, 3], dtype="float64")
print(float_arr, float_arr.dtype)

# Creating new array with a specific type from existing array
int_arr = float_arr.astype(np.int64)
print(int_arr, int_arr.dtype)


['1' '2' '3'] <U1
[1. 2. 3.] float64
[1 2 3] int64


## 7. Operations on numpy arrays

In [9]:
#1.Reshaping
arr = np.array([1,2,3,4,5,6])
print(arr.shape)

reshaped_arr = arr.reshape((2,3))
print(reshaped_arr, reshaped_arr.shape)

flattened = reshaped_arr.flatten()
print(flattened, flattened.shape)

(6,)
[[1 2 3]
 [4 5 6]] (2, 3)
[1 2 3 4 5 6] (6,)


In [10]:
# 2. Indexing
# 1-D array indexing
arr = np.array([10, 20, 30, 40, 50])
print(arr[0])  # First element
print(arr[-1]) # Last element

# 2-D array indexing
arr = np.array([[1,2,3],[4,5,6],[7,8,9]])
print(arr[0, 0])  # First row, first column
print(arr[1][2])  # Second row, third column
print(arr[2, :])  # Third row

# Fancy indexing
arr = np.array([10, 20, 30, 40, 50])
indices = [0, 2, 4]
print(arr[indices])  # Output: [10 30 50]
print(arr[arr>30])  # Output: [40 50]
print(arr[arr%4==0])  # Output: [20 40]

10
50
1
6
[7 8 9]
[10 30 50]
[40 50]
[20 40]


In [11]:
# 3. Slicing
# 1-D array slicing
arr = np.array([10, 20, 30, 40, 50])
print(arr[1:4])  # Elements from index 1 to 3
print(arr[:3])   # First three elements
print(arr[2:])   # Elements from index 2 to the end

# 2-D array slicing
arr = np.array([[1,2,3],[4,5,6],[7,8,9]])
print(arr[0:2, 1:3])  # First two rows, columns 1 and 2
print(arr[:, 1])      # All rows, second column 
print(arr[1, :])      # Second row, all columns
print(arr[0,::-1])   # First row, reversed order

[20 30 40]
[10 20 30]
[30 40 50]
[[2 3]
 [5 6]]
[2 5 8]
[4 5 6]
[3 2 1]


### Copy v/s View
Slicing a list returns a copy but slicing a NumPy array returns a view — for efficiency. View is like a shallow copy that shares the same data as the original array, so no duplication happens.
- **Views** are **fast** and **memory-efficient** (no data duplication).
- **Copies** are **safe** but **slower** and use more memory.


In [12]:
# Sliced List is a COPY
py_list = [1, 2, 3, 4, 5]
list_slice = py_list[1:4]  # Elements from index 1 to 3
print(list_slice)  # Output: [2, 3, 4]
list_slice[0] = 7
print(py_list)  # Output: [1, 2, 3, 4, 5] - original list remains unchanged
print(list_slice)  # Output: [7, 3, 4] - modified slice

[2, 3, 4]
[1, 2, 3, 4, 5]
[7, 3, 4]


In [13]:
# Sliced Numpy Array is a VIEW
np_array = np.array([1, 2, 3, 4, 5])
array_slice = np_array[1:4]  # Elements from index 1 to 3
print(array_slice)  # Output: [2 3 4]
array_slice[0] = 7
print(np_array)  # Output: [1 7 3 4 5]
print(array_slice)  # Output: [7 3 4] - modified slice

[2 3 4]
[1 7 3 4 5]
[7 3 4]


In [14]:
# Creating a COPY for Array
copy_array = np_array[1:4].copy()  # Create a copy of the slice
print(copy_array)  # Output: [7 3 4]
copy_array[0] = 10
print(np_array)  # Output: [1 7 3 4 5]  
print(copy_array)  # Output: [10 3 4] - modified copy

[7 3 4]
[1 7 3 4 5]
[10  3  4]


## 8.Multidimensional Arrays
![](https://i.imgur.com/mg8O3kd.png)

**Multi-dimensional arrays** in NumPy are the foundation of most scientific and machine-learning work.
A NumPy array can have **any number of dimensions** (1D, 2D, 3D & so on). Each dimension is called an **axis.**
- 1D array has 1 axis (axis0).
- 2D array has 2 axes (axis0 = rows, axis1 = columns)
- 3D array has 3 axes (axis0 = depth/layer, axis1 = rows in each layer, axis2 = columns in each layer)

💡 **Usage** of multi-dimensional arrays in Practice:
- **Grayscale Image as 2D array —** A grayscale image has height × width pixels. Each pixel has a single intensity value (0–255 for 8-bit images).
- **Color Images as 3D array —** A color image (RGB) has height × width × channels. The 3 channels are RGB (Red, Green, Blue) & each channel is a 2D array of pixel intensities (0–255).



In [15]:
# 1D Array
arr1D = np.array([7,2,9,10])
print(arr1D, arr1D.shape, arr1D.ndim)

# 2D Array
arr2D = np.array([[5.2, 3.0, 4.5], [9.1, 0.1, 0.3]])
print(arr2D, arr2D.shape, arr2D.ndim)

# 3D Array
arr3D = np.array([[[1,2,3],[4,5,6],[7,8,9],[10,11,12]],[[1,2,3],[4,5,6],[7,8,9],[10,11,12]],[[1,2,3],[4,5,6],[7,8,9],[10,11,12]]])
print(arr3D, arr3D.shape, arr3D.ndim)

[ 7  2  9 10] (4,) 1
[[5.2 3.  4.5]
 [9.1 0.1 0.3]] (2, 3) 2
[[[ 1  2  3]
  [ 4  5  6]
  [ 7  8  9]
  [10 11 12]]

 [[ 1  2  3]
  [ 4  5  6]
  [ 7  8  9]
  [10 11 12]]

 [[ 1  2  3]
  [ 4  5  6]
  [ 7  8  9]
  [10 11 12]]] (3, 4, 3) 3


### Operations along Axes
We can also perform certain **operations along a specific axis** in an array.


In [16]:
arr2D = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(arr2D)  # Output: 2D array
print(np.sum(arr2D))  # Sum of all elements
print(np.sum(arr2D, axis=0))  # Sum of each column
print(np.sum(arr2D, axis=1))  # Sum of each row

print(arr2D[0:2,1:2])  # Slicing rows 0 and 1, and column 1


[[1 2 3]
 [4 5 6]
 [7 8 9]]
45
[12 15 18]
[ 6 15 24]
[[2]
 [5]]


### Operations on 3D Arrays

In [17]:
arr3D = np.array([[[1,2,3],[4,5,6]],[[7,8,9],[10,11,12]]])
print(arr3D, arr3D.ndim, arr3D.shape)

#indexing 
print(arr3D[0,1,1])  # Accessing the element at position [0,1,1]
print(arr3D[1,1,2])  # Accessing the element at position [1,1,2]

print(arr3D[:,:,0])  # Accessing all elements in the first column across all 2D arrays
print(arr3D[:,0,:])  # Accessing all elements in the first row across all 2D arrays

arr3D[:,0,:] = 99
print(arr3D)  # Printing the array after modification

[[[ 1  2  3]
  [ 4  5  6]]

 [[ 7  8  9]
  [10 11 12]]] 3 (2, 2, 3)
5
12
[[ 1  4]
 [ 7 10]]
[[1 2 3]
 [7 8 9]]
[[[99 99 99]
  [ 4  5  6]]

 [[99 99 99]
  [10 11 12]]]


## 9.Data Types in NumPy
Every NumPy array has a **single data type** (homogeneous arrays) stored in the .dtype attribute. Common data types:
- **Integer literals:** int32, int64
- **Floating literals:** float32, float64
- **Boolean:** bool
- **Complex numbers:** complex64, complex128
- **String:** S (byte-str) & U (unicode-str)
- **Object:** generic python objects — object


In [18]:
# Common Data Types
arr  = np.array([1, 2, 3, 4, 5])
arr2 = np.array([1.0, 2.0, 3.0])
arr3 = np.array(["hello", "world", "prime", "ai/ml"])

print(arr.dtype)   # int64
print(arr2.dtype)  # float64
print(arr3.dtype)  # U

# Complex Numbers
arr1 = np.array([2 + 3j])
arr2 = np.array([5 + 8j])
print(arr1 + arr2)
print(arr2 - arr1)

# Objects
arr_obj = np.array(["hello", {1, 2, 3}, 3.14])
print(arr_obj, arr_obj.dtype)

# Changing the data type
new_arr = arr.astype("float64")
print(new_arr, new_arr.dtype)

new_arr = np.array([1, 2, 3, 4, 5], dtype="float64")
print(new_arr, new_arr.dtype)


int64
float64
<U5
[7.+11.j]
[3.+5.j]
['hello' {1, 2, 3} 3.14] object
[1. 2. 3. 4. 5.] float64
[1. 2. 3. 4. 5.] float64


## 10.Vectorization & Broadcasting

**Vectorization**
- Vectorization means performing **operations on entire arrays at once** without explicit Python loops.
    -   NumPy uses **C-level implementations internally** → much faster than Python loops.
    -   Makes code shorter, cleaner, and faster.


In [19]:
arr = np.array([1, 2, 3, 4, 5])
sq_array = arr ** 2
print(sq_array)  # Output: [ 1  4  9 16 25] Square of all nums

arr2 = np.array([6, 7, 8, 9, 10])
print(arr + arr2)  # Output: [ 7  9 11 13 15] # Sum of two arrays

[ 1  4  9 16 25]
[ 7  9 11 13 15]


## Broadcasting
Broadcasting allows NumPy to **automatically expand arrays of different shapes** so that arithmetic operations can be performed. It's basically scaling arrays without using extra memory.
- No need to manually reshape arrays.
- Useful for **combining arrays of different dimensions.**

**Broadcasting Rules**
Broadcasting can only take place when arrays are of compatible shape. NumPy compares shapes from right to left. All dimensions must either be:
- **Equal,** or
- **1,** or
- **Missing** (smaller array can be "stretched").


In [20]:
#Broadcasting with a Scalar
arr = np.array([1, 2, 3, 4, 5])
scalar = 2
print(arr * scalar)  # Output: [ 2  4  6  8 10] Multiplying array with scalar
print(arr + scalar)  # Output: [ 3  4  5  6  7] Adding scalar to array

#Broadcasting with a Vector
arr1D = np.array([1,2,3])
arr2D = np.array([[1,2,3],[4,5,6],[7,8,9]])
print(arr1D + arr2D)  # Output: [[ 2  4  6] [ 5  7  9] [ 8 10 12]] Adding vector to each row of the matrix

[ 2  4  6  8 10]
[3 4 5 6 7]
[[ 2  4  6]
 [ 5  7  9]
 [ 8 10 12]]


### Normalization
- Normalizing an array typically involves scaling the values to a specific range, often [0, 1] or [-1, 1]. This can be done using various techniques, such as min-max normalization or z-score normalization.
- A Common broacasting example is Vector Normalization - Transforming an array  to have mean = 0 and Standard Deviation = 0
- For each element xᵢ in vector:   xᵢ(normalised) = (xᵢ − μ) / σ
Where μ = mean of the vector and σ = standard deviation.


In [21]:
arr = np.array([[1,2],[3,4]])
print("2D Array:\n", arr)

# Mean
mean = arr.sum()/arr.size
print("Mean:", mean)

# Variance
variance = ((arr - mean) ** 2).sum() / arr.size
print("Variance:", variance)

# Standard Deviation
std_dev = np.sqrt(variance)
print("Standard Deviation:", std_dev)



2D Array:
 [[1 2]
 [3 4]]
Mean: 2.5
Variance: 1.25
Standard Deviation: 1.118033988749895


In [22]:
#Standard Vector Normalization
arr = np.array([[1, 2], [3, 4]])
mean = np.mean(arr)
std_dev = np.std(arr)
normalized_arr = (arr - mean) / std_dev
print("Normalized Array:\n", normalized_arr)

Normalized Array:
 [[-1.34164079 -0.4472136 ]
 [ 0.4472136   1.34164079]]


## 11. Useful Mathematic Functions
NumPy provides a **wide range of built-in mathematical functions** that are highly optimized and operate **element-wise on arrays.**

**Aggregation Functions**
- sum()  — returns sum of all elements
- prod() — returns product of all elements
- min()  — returns minimum value
- max()  — returns maximum value
- argmin() — returns index of min value
- argmax() — returns index of max value
- mean()   — returns mean (average)
- median() — returns median
- std()   — returns standard deviation
- var()   — returns variance



In [23]:
arr = np.array([1, 2, 3, 4, 5])
print(np.sum(arr))  # Output: 15
print(np.prod(arr))  # Output: 120
print(np.max(arr))  # Output: 5
print(np.min(arr))  # Output: 1 
print(np.argmax(arr))  # Output: 4 (index of max element)
print(np.argmin(arr))  # Output: 0 (index of min element)
print(np.mean(arr))  # Output: 3.0 (mean of the array)
print(np.median(arr))  # Output: 3.0 (median of the array)
print(np.std(arr))  # Output: 1.4142135623730951 (standard deviation of the array)
print(np.var(arr))  # Output: 2.0 (variance of the array)

15
120
5
1
4
0
3.0
3.0
1.4142135623730951
2.0


**Power Functions**
- square()   — returns square of all elements
- sqrt()     — returns square root of all elements
- pow(a, b)  — returns a^b


In [24]:
arr = np.array([1, 2, 3, 4, 5])
print(np.square(arr))  # Output: [ 1  4  9 16 25] (square of each element  )
print(np.sqrt(arr))  # Output: [1.         1.41421356 1.73205081 2.         2.23606798] (square root of each element)
print(np.pow(arr, 3))  # Output: [ 1  8 27 64 125] (each element raised to the power of 3)

[ 1  4  9 16 25]
[1.         1.41421356 1.73205081 2.         2.23606798]
[  1   8  27  64 125]


**Log & Exponential Functions**
- log()    — returns natural log
- log10()  — returns log base 10
- log2()   — returns log base 2
- exp(x)   — returns eˣ


In [25]:
arr = np.array([1, 2, 3, 4, 5])

print(np.log(arr))  # Output: [0.         0.69314718 1.09861229 1.38629436 1.60943791] (natural logarithm of each element)
print(np.log10(arr))  # Output: [0.         0.30103 0.47712125 0.60205999 0.69897] (base 10 logarithm of each element)
print(np.log2(arr))  # Output: [0.         1.         1.5849625  2.         2.32192809] (base 2 logarithm of each element)      
print(np.exp(arr))  # Output: [  2.71828183   7.3890561   20.08553692  54.59815003 148.4131591 ] (exponential of each element)

[0.         0.69314718 1.09861229 1.38629436 1.60943791]
[0.         0.30103    0.47712125 0.60205999 0.69897   ]
[0.         1.         1.5849625  2.         2.32192809]
[  2.71828183   7.3890561   20.08553692  54.59815003 148.4131591 ]


**Rounding Functions**
- round()    — rounds off to nearest value
- ceil()     — rounds up
- floor()    — rounds down
- trunc(x)   — truncates (removes) the fractional part


In [26]:
print(np.round(2.678))  # 3.0
print(np.floor(2.678))  # 2.0
print(np.ceil(2.678))   # 3.0
print(np.trunc(2.678))  # 2.0

3.0
2.0
3.0
2.0


**Miscellaneous Functions**
- abs()    — returns absolute value
- sort()   — returns sorted (increasing order) values
- unique() — returns unique values


In [27]:
arr = np.array([1, 2, -5, 3, 8, -4, 2, 5])

print(np.abs(arr))     # [1 2 5 3 8 4 2 5]
print(np.sort(arr))    # [-5 -4 1 2 2 3 5 8]
print(np.unique(arr))  # [-5 -4 1 2 3 5 8]


[1 2 5 3 8 4 2 5]
[-5 -4  1  2  2  3  5  8]
[-5 -4  1  2  3  5  8]
